 > ### ***Install Kaggle and download dataset*** ###

In [2]:
!pip install -q kaggle

from google.colab import files

files.upload()


!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d asaniczka/amazon-kindle-books-dataset-2023-130k-books

!unzip -q amazon-kindle-books-dataset-2023-130k-books.zip

print("Dataset downloaded and extracted successfully!")


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/asaniczka/amazon-kindle-books-dataset-2023-130k-books
License(s): ODC Attribution License (ODC-By)
  0% 0.00/9.11M [00:00<?, ?B/s]
100% 9.11M/9.11M [00:00<00:00, 900MB/s]
Dataset downloaded and extracted successfully!


> ### ***Import Necessary Libraries*** ###


In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os



> ### ***Import & Load Data*** ###


In [4]:
df = pd.read_csv("kindle_data-v2.csv")

print("Data loaded successfully!")

Data loaded successfully!


In [5]:
print(f"Total books before cleaning: {len(df)}")

Total books before cleaning: 133102


In [6]:
print("Columns:", df.columns.tolist())


Columns: ['asin', 'title', 'author', 'soldBy', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'isKindleUnlimited', 'category_id', 'isBestSeller', 'isEditorsPick', 'isGoodReadsChoice', 'publishedDate', 'category_name']


> ### ***Initial Data Exploration*** ###

In [7]:
df.head(3)


,asin,title,author,soldBy,imgUrl,productURL,stars,reviews,price,isKindleUnlimited,category_id,isBestSeller,isEditorsPick,isGoodReadsChoice,publishedDate,category_name
0,B00TZE87S4,Adult Children of Emotionally Immature Parents...,Lindsay C. Gibson,Amazon.com Services LLC,https://m.media-amazon.com/images/I/713KZTsaYp...,https://www.amazon.com/dp/B00TZE87S4,4.8,0,9.99,False,6,True,False,False,2015-06-01,Parenting & Relationships
1,B08WCKY8MB,"From Strength to Strength: Finding Success, Ha...",Arthur C. Brooks,Penguin Group (USA) LLC,https://m.media-amazon.com/images/I/A1LZcJFs9E...,https://www.amazon.com/dp/B08WCKY8MB,4.4,0,16.99,False,6,False,False,False,2022-02-15,Parenting & Relationships
2,B09KPS84CJ,Good Inside: A Guide to Becoming the Parent Yo...,Becky Kennedy,HarperCollins Publishers,https://m.media-amazon.com/images/I/71RIWM0sv6...,https://www.amazon.com/dp/B09KPS84CJ,4.8,0,16.99,False,6,False,True,False,2022-09-13,Parenting & Relationships


> ### ***Data Preprocessing*** ###

#### ***`Clean & Rename data`*** ####

In [8]:
def get_track(cat):
    cat = str(cat).lower()
    if any(word in cat for word in ['math', 'statistics', 'probability', 'algebra', 'calculus', 'geometry']):
        return 'Mathematics & Statistics'
    elif any(word in cat for word in ['machine learning', 'deep learning', 'data science', 'artificial intelligence', 'python', 'programming', 'software', 'algorithm']):
        return 'Machine Learning & Programming'
    elif any(word in cat for word in ['computer', 'technology', 'network', 'security', 'database', 'cloud']):
        return 'Computer Science & Technology'
    elif any(word in cat for word in ['education', 'teaching', 'learning', 'pedagogy']):
        return 'Education & Teaching'
    else:
        return 'Science & Engineering'

df["track"] = df["category_name"].apply(get_track)
df["rating"] = df["stars"]
df["text_for_similarity"] = (
    df["title"].fillna("") + " " +
    df["author"].fillna("") + " " +
    df["category_name"].fillna("")
)
print("Extra columns (track, rating, text_for_similarity) created.")


Extra columns (track, rating, text_for_similarity) created.




> #### ***`Final Dataset Preview`*** ####



In [9]:
df[["title", "author", "category_name", "track", "rating"]].head()


,title,author,category_name,track,rating
0,Adult Children of Emotionally Immature Parents...,Lindsay C. Gibson,Parenting & Relationships,Science & Engineering,4.8
1,"From Strength to Strength: Finding Success, Ha...",Arthur C. Brooks,Parenting & Relationships,Science & Engineering,4.4
2,Good Inside: A Guide to Becoming the Parent Yo...,Becky Kennedy,Parenting & Relationships,Science & Engineering,4.8
3,Everything I Know About Love: A Memoir,Dolly Alderton,Parenting & Relationships,Science & Engineering,4.2
4,The Seven Principles for Making Marriage Work:...,John Gottman,Parenting & Relationships,Science & Engineering,4.7


Build final books dataframe with only required columns

In [10]:
tech_tracks = [
    'Machine Learning & Programming',
    'Computer Science & Technology',
    'Mathematics & Statistics',
    'Education & Teaching'
]

books = df[[
    "asin", "title", "author", "category_name", "track", "rating",
    "reviews", "imgUrl", "productURL", "text_for_similarity"
]].copy()

books.rename(columns={"imgUrl": "image", "productURL": "link"}, inplace=True)
books.dropna(subset=["title", "link"], inplace=True)
books = books[(books["reviews"] >= 5) & (books["rating"] >= 3)]
books = books[books["track"].isin(tech_tracks)]

print(f"Number of TECH books after cleaning: {len(books)}")
print(books["track"].value_counts())
books[["title", "author", "track", "rating", "image", "link"]].head()


Number of TECH books after cleaning: 9237
track
Computer Science & Technology    4783
Education & Teaching             4454
Name: count, dtype: int64


,title,author,track,rating,image,link
38157,Chip War: The Fight for the World's Most Criti...,Chris Miller,Computer Science & Technology,4.7,https://m.media-amazon.com/images/I/71hMs9v7P6...,https://www.amazon.com/dp/B09RX5F238
38158,"Glossy: Ambition, Beauty, and the Inside Story...",Marisa Meltzer,Computer Science & Technology,3.8,https://m.media-amazon.com/images/I/61CPMk0tzD...,https://www.amazon.com/dp/B0BHTQRR71
38159,Broken Money: Why Our Financial System is Fail...,Lyn Alden,Computer Science & Technology,4.8,https://m.media-amazon.com/images/I/81WR94BwU7...,https://www.amazon.com/dp/B0CGNVNXK2
38160,(ISC)2 CISSP Certified Information Systems Sec...,Mike Chapple,Computer Science & Technology,4.7,https://m.media-amazon.com/images/I/61W0HBUd+h...,https://www.amazon.com/dp/B097NHJK9Q
38161,CompTIA Security+ Get Certified Get Ahead: SY0...,Darril Gibson,Computer Science & Technology,4.7,https://m.media-amazon.com/images/I/81sqkapJ3Y...,https://www.amazon.com/dp/B09237T9ZB


Number Of Books After Cleaning

In [11]:
print(f"Number of TECH books after cleaning: {len(books)}")
print(books["track"].value_counts())


Number of TECH books after cleaning: 9237
track
Computer Science & Technology    4783
Education & Teaching             4454
Name: count, dtype: int64


Preview final books columns

In [12]:
books[["title", "author", "track", "rating", "image", "link"]].head()


,title,author,track,rating,image,link
38157,Chip War: The Fight for the World's Most Criti...,Chris Miller,Computer Science & Technology,4.7,https://m.media-amazon.com/images/I/71hMs9v7P6...,https://www.amazon.com/dp/B09RX5F238
38158,"Glossy: Ambition, Beauty, and the Inside Story...",Marisa Meltzer,Computer Science & Technology,3.8,https://m.media-amazon.com/images/I/61CPMk0tzD...,https://www.amazon.com/dp/B0BHTQRR71
38159,Broken Money: Why Our Financial System is Fail...,Lyn Alden,Computer Science & Technology,4.8,https://m.media-amazon.com/images/I/81WR94BwU7...,https://www.amazon.com/dp/B0CGNVNXK2
38160,(ISC)2 CISSP Certified Information Systems Sec...,Mike Chapple,Computer Science & Technology,4.7,https://m.media-amazon.com/images/I/61W0HBUd+h...,https://www.amazon.com/dp/B097NHJK9Q
38161,CompTIA Security+ Get Certified Get Ahead: SY0...,Darril Gibson,Computer Science & Technology,4.7,https://m.media-amazon.com/images/I/81sqkapJ3Y...,https://www.amazon.com/dp/B09237T9ZB


> ### ***Build TF-IDF Vectors & Cosine Similarity Matrix*** ###

In [13]:
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(books['text_for_similarity'])
cosine_sim = cosine_similarity(tfidf_matrix)

print("Recommendation engine ready!")


Recommendation engine ready!


> ### ***Build and Test Recommendation Function*** ###

In [14]:
def recommend(title, top_k=8):
    # Reset index to ensure clean 0-N indexing
    books_reset = books.reset_index(drop=True)

    if title not in books_reset['title'].values:
        print("Book not found")
        return

    # Use reset index (0 to len(books)-1)
    idx = books_reset[books_reset['title'] == title].index[0]

    # Now idx is guaranteed within cosine_sim bounds
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_k+1]

    print(f"\nRecommended because you viewed:\n→ {title}\n")
    for i, score in sim_scores:
        b = books_reset.iloc[i]
        print(f"• {b['title']}")
        print(f"   Author: {b['author']} | Track: {b['track']} | Rating: {b['rating']}/5")
        print(f"   Image → {b['image']}")
        print(f"   Link  → {b['link']}\n")


In [15]:
recommend("Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow")
recommend("Introduction to Probability")
recommend("Python Crash Course")
recommend("Clean Code")



Recommended because you viewed:
→ Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow

• Python Machine Learning: Machine Learning and Deep Learning with Python, scikit-learn, and TensorFlow 2, 3rd Edition
   Author: Sebastian Raschka | Track: Computer Science & Technology | Rating: 4.5/5
   Image → https://m.media-amazon.com/images/I/81QR4kiRN8L._AC_UY218_.jpg
   Link  → https://www.amazon.com/dp/B07VBLX2W7

• Machine Learning with PyTorch and Scikit-Learn: Develop machine learning and deep learning models with Python
   Author: Sebastian Raschka | Track: Computer Science & Technology | Rating: 4.5/5
   Image → https://m.media-amazon.com/images/I/81rACoCjzqL._AC_UY218_.jpg
   Link  → https://www.amazon.com/dp/B09NW48MR1

• Deep Learning with TensorFlow and Keras: Build and deploy supervised, unsupervised, deep, and reinforcement learning models, 3rd Edition
   Author: Amita Kapoor | Track: Computer Science & Technology | Rating: 4.6/5
   Image → https://m.media-amazon.

> ### ***Save Model & Data for Deployment*** ###

In [16]:
# Save cleaned DataFrame and model files for FastAPI backend
books.to_csv('FINAL_BOOKS_CLEAN.csv', index=False)

import pickle
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(cosine_sim, open('similarity_sparse.pkl', 'wb'))

print("All core files saved for FastAPI backend:")
print("- FINAL_BOOKS_CLEAN.csv")
print("- vectorizer.pkl")
print("- similarity_sparse.pkl")


All core files saved for FastAPI backend:
- FINAL_BOOKS_CLEAN.csv
- vectorizer.pkl
- similarity_sparse.pkl


> ### ***Download Files (Google Colab)*** ###

In [17]:
from google.colab import files

files.download("FINAL_BOOKS_CLEAN.csv")
files.download("vectorizer.pkl")
files.download("similarity_sparse.pkl")

print("Download complete.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete.
